# Detailed Solutions Notebook — Regularization for a Mincer Wage Equation

The fully worked answer key to `econ_02_Skeleton_Practice_Notebook.ipynb`, with reasoning after every checkpoint.

## Part 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('wage_survey.csv')
y = df['log_hourly_wage']
features = df.drop(columns=['log_hourly_wage', 'hourly_wage'])
predictors = features.columns
print(df.shape, len(predictors), 'features')

**Why exclude `hourly_wage` from features:** it's a deterministic transform of the target (`log_hourly_wage = log(hourly_wage)`), so leaving it in the feature set would let the model "predict" the target almost perfectly by reversing a logarithm — a textbook data leakage bug.

## Part 1 — EDA

In [ ]:
corr = features.assign(log_hourly_wage=y).corr()
plt.figure(figsize=(12, 9))
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.tight_layout()
plt.show()
corr['log_hourly_wage'].sort_values(ascending=False).head(8)

**Answer:** `age`, `education_years`, `experience_years`, `tenure_years`, and `experience_sq` are the strongest correlates of log wages — exactly the classic Mincer-equation variables. `union_member` and `urban` also show meaningful positive correlation.

In [ ]:
print(features[['age', 'education_years', 'experience_years']].corr())
print('corr(experience, experience_sq):', np.corrcoef(features['experience_years'], features['experience_sq'])[0, 1])

**Why this matters:** `age`, `education_years`, and `experience_years` are correlated by construction — in the data-generating process, `experience ≈ age − education − 6`. When all three are included as separate regressors, OLS has to divide credit for wage growth among three variables that are largely telling the same story, which inflates the variance of each individual coefficient (classic multicollinearity). `experience_sq` is *exactly* a function of `experience_years` (correlation ~0.97 here, not exactly 1.0 only because of the small measurement noise we added), so including both is almost the textbook definition of a collinear pair — necessary for capturing the concave (diminishing-returns) shape of experience's effect, but something a regularized model will need to handle gracefully.

## Part 2 — Preprocessing

In [ ]:
from sklearn.preprocessing import StandardScaler
X = StandardScaler().fit_transform(features)

## Part 3 — Train/test split

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Part 4 — Baseline OLS

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

ols = LinearRegression().fit(X_train, y_train)
print('Train MSE:', mean_squared_error(y_train, ols.predict(X_train)), '| R2:', r2_score(y_train, ols.predict(X_train)))
print('Test  MSE:', mean_squared_error(y_test, ols.predict(X_test)), '| R2:', r2_score(y_test, ols.predict(X_test)))

In [ ]:
coef = pd.Series(ols.coef_, predictors).sort_values()
coef.plot(kind='bar', title='OLS coefficients (standardized)')
plt.axhline(0, color='k', linewidth=0.8)
plt.tight_layout()
plt.show()

**Observations:** training and test MSE/R² are close here because 2,000 observations comfortably support 19 regressors — a reminder that overfitting severity scales with the *ratio* of features to observations, not just the feature count. Still, the "noise" controls (`sibling_count`, `commute_minutes`, `coffee_cups_per_day`, `household_size`, `savings_rate_pct`) show small nonzero coefficients purely from sampling variation, and the age/education/experience trio's coefficients look less individually interpretable than a clean four-variable Mincer equation would produce — both symptoms regularization is designed to address.

## Part 5 — Ridge regression

In [ ]:
from sklearn.linear_model import Ridge, RidgeCV

ridge1 = Ridge(alpha=1.0).fit(X_train, y_train)
print('Ridge(alpha=1) test MSE:', mean_squared_error(y_test, ridge1.predict(X_test)))

alphas = np.logspace(-3, 4, 100)
ridge_cv = RidgeCV(alphas=alphas, cv=5).fit(X_train, y_train)
print('Best alpha:', ridge_cv.alpha_)
print('Test MSE at best alpha:', mean_squared_error(y_test, ridge_cv.predict(X_test)))

In [ ]:
train_mse = [mean_squared_error(y_train, Ridge(alpha=a).fit(X_train, y_train).predict(X_train)) for a in alphas]
test_mse = [mean_squared_error(y_test, Ridge(alpha=a).fit(X_train, y_train).predict(X_test)) for a in alphas]
plt.plot(alphas, train_mse, label='Train MSE')
plt.plot(alphas, test_mse, label='Test MSE')
plt.axvline(ridge_cv.alpha_, linestyle='--', color='k', label='CV-selected alpha')
plt.xscale('log')
plt.legend()
plt.show()

**Reading the plot:** test MSE is fairly flat over a wide range of small-to-moderate alphas (since OLS wasn't badly overfitting to begin with) and only rises once alpha gets large enough to meaningfully shrink genuinely informative coefficients like education and experience — that's the underfitting regime.

## Part 6 — Lasso and variable selection

In [ ]:
from sklearn.linear_model import LassoCV

lasso_cv = LassoCV(alphas=np.logspace(-4, 1, 100), cv=5, max_iter=10000).fit(X_train, y_train)
print('Best alpha:', lasso_cv.alpha_)
print('Test MSE:', mean_squared_error(y_test, lasso_cv.predict(X_test)))

In [ ]:
coef_lasso = pd.Series(lasso_cv.coef_, predictors).sort_values()
coef_lasso.plot(kind='bar', title='Lasso coefficients (tuned)')
plt.axhline(0, color='k', linewidth=0.8)
plt.tight_layout()
plt.show()

zeroed = coef_lasso[coef_lasso == 0]
print(f'{len(zeroed)} of {len(coef_lasso)} features zeroed:', list(zeroed.index))

**Comparison:** Lasso typically zeroes out most of the deliberately-irrelevant controls (`sibling_count`, `commute_minutes`, `household_size`, `coffee_cups_per_day`, `savings_rate_pct`, and often `south_region`/`married`, whose true effects were small in the data-generating process). It usually keeps one of `age` / `experience_years` (they carry overlapping information) rather than both — a direct illustration of L1's "arbitrary pick one from a correlated group" behavior. This is a real feature-selection success, but note it is a *statistical* selection, not evidence that the dropped correlated twin has zero true economic effect.

## Part 7 — Elastic Net

In [ ]:
from sklearn.linear_model import ElasticNetCV

enet_cv = ElasticNetCV(
    alphas=np.logspace(-4, 1, 50), l1_ratio=[.1, .3, .5, .7, .9], cv=5, max_iter=10000
).fit(X_train, y_train)
print('Best alpha:', enet_cv.alpha_, '| l1_ratio:', enet_cv.l1_ratio_)
print('Test MSE:', mean_squared_error(y_test, enet_cv.predict(X_test)))

## Part 8 — Model comparison

In [ ]:
models = {'OLS': ols, 'Ridge (tuned)': ridge_cv, 'Lasso (tuned)': lasso_cv, 'Elastic Net (tuned)': enet_cv}
rows = []
for name, m in models.items():
    pred = m.predict(X_test)
    rows.append({'Model': name, 'Test MSE': mean_squared_error(y_test, pred),
                 'Test R2': r2_score(y_test, pred),
                 'Nonzero coefs': int(np.sum(np.abs(m.coef_) > 1e-8))})
pd.DataFrame(rows).set_index('Model').round(4)

**Findings:** with a comfortable observations-to-features ratio, all four models land close together on test MSE/R² — regularization here mainly buys interpretability (via Lasso's sparsity) rather than a large predictive-accuracy win. That balance flips in more feature-rich economics settings (e.g., growth regressions with dozens of country-level indicators and only 60-100 country observations), where regularization can meaningfully outperform OLS on held-out data.

## Part 9 — Prediction vs. causal inference (sample answers)

1. **Why Lasso/Ridge coefficients are biased toward zero:** both add a penalty term that the optimizer must minimize alongside the data-fit loss, so the "optimal" coefficient under the combined objective is smaller in magnitude than the unbiased OLS coefficient would be. This is acceptable for prediction (a slightly-too-small coefficient can still combine with other slightly-shrunk coefficients to produce accurate predictions) but is a direct problem if the number itself is the object of interest — a shrunk union-wage-premium coefficient understates the true premium by construction, in a way that doesn't average out.

2. **Double-selection Lasso, informally:** run Lasso twice — once predicting the outcome from the controls, once predicting the variable of interest (e.g., union membership) from the controls — and keep the union of controls either Lasso judged relevant. Then run one more, ordinary OLS regression of the outcome on the variable of interest plus that selected control set. Because OLS (not Lasso) produces the final coefficient, and because using *both* Lasso runs' selected controls guards against dropping a control that matters for confounding even if it's a weak direct predictor of the outcome, the resulting estimate is approximately unbiased under standard assumptions — this is the intuition behind the Frisch–Waugh–Lovell theorem applied with penalized first stages.

3. **Which number to defend to a policymaker:** the OLS-after-Lasso-selection coefficient, and say so explicitly — "we used Lasso to choose which of 19 candidate controls to include, then estimated the union premium with ordinary least squares on that selected specification." Presenting the raw Lasso coefficient as a causal effect invites a fair challenge, since by construction it is shrunk toward zero relative to the unbiased estimate; presenting the properly-staged OLS estimate keeps the (defensible) benefit of principled control selection without the shrinkage bias.